# ДЗ 5. CTE-pipeline, recursive CTE, Materialized View

Датасет: Brazilian E-Commerce Public Dataset by Olist
https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

Таблиці: customers, orders, order_items, products, sellers.

## Завдання 1. Setup і typed-схема

pgserver ставлю окремо в папку `pg_pkg`, бо через звичайний pip у Colab він не встановлюється (є тільки для Python до 3.12).

In [1]:
%pip install -q psycopg2-binary sqlalchemy pandas kagglehub fasteners platformdirs psutil
%pip install -q --upgrade --no-deps --only-binary=:all: --python-version 3.12 --target pg_pkg pgserver

In [2]:
import os
import sys
import tempfile
from urllib.request import urlretrieve

sys.path.insert(0, os.path.abspath('pg_pkg'))

import kagglehub
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

Драйвер `psycopg2` пишу в адресі явно, інакше SQLAlchemy шукає psycopg 3.

In [3]:
pg_dir = os.path.join(tempfile.gettempdir(), 'hw5_pg')
pg = pgserver.get_server(pg_dir, cleanup_mode='stop')
db_uri = pg.get_uri().replace('postgresql://', 'postgresql+psycopg2://')
engine = create_engine(db_uri, future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


`run` виконує SQL з комітом і повертає DataFrame, `explain` друкує план.

In [4]:
def run(query):
    with engine.begin() as conn:
        res = conn.execute(text(query))
        if res.returns_rows:
            return pd.DataFrame(res.fetchall(), columns=list(res.keys()))
        print('ok, rows affected:', res.rowcount)


def explain(query):
    plan = run('EXPLAIN (ANALYZE, BUFFERS) ' + query)
    print('\n'.join(plan.iloc[:, 0]))

### Завантаження CSV

Спочатку KaggleHub. Якщо не вийде, качаю ті самі файли з офіційного GitHub Olist у `data/olist`.

In [5]:
files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
}

try:
    dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
except Exception as e:
    print('KaggleHub не спрацював:', e)
    dataset_dir = 'data/olist'
    os.makedirs(dataset_dir, exist_ok=True)
    base_url = 'https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/'
    for file_name in files.values():
        path = os.path.join(dataset_dir, file_name)
        if not os.path.exists(path):
            urlretrieve(base_url + file_name, path)

print(dataset_dir)

KaggleHub не спрацював: HTTPSConnectionPool(host='api.kaggle.com', port=443): Max retries exceeded with url: /v1/datasets.DatasetApiService/GetDataset (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))
data/olist


Raw-шар:

In [6]:
raw_counts = {}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
    raw_counts[table_name] = len(df)
    print(table_name, df.shape)

olist_customers_raw (99441, 5)


olist_orders_raw (99441, 8)


olist_order_items_raw (112650, 7)


olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)


Typed-шар:

In [7]:
run("""
DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;
DROP TABLE IF EXISTS hw5_referrals     CASCADE;
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders      CASCADE;
DROP TABLE IF EXISTS olist_customers   CASCADE;
DROP TABLE IF EXISTS olist_products    CASCADE;
DROP TABLE IF EXISTS olist_sellers     CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);
""")

ok, rows affected: -1


raw -> typed. Спочатку таблиці без FK. У CSV товарів колонки з помилкою `lenght`.

In [8]:
run("""
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;
""")

ok, rows affected: 112650


### Smoke-test

In [9]:
typed = run("""
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
ORDER BY table_name;
""")

typed['raw_rows'] = typed['table_name'].map(lambda t: raw_counts['olist_' + t + '_raw'])
typed['diff'] = typed['raw_rows'] - typed['n_rows']
typed

,table_name,n_rows,raw_rows,diff
0,customers,99441,99441,0
1,order_items,112650,112650,0
2,orders,99441,99441,0
3,products,32951,32951,0
4,sellers,3095,3095,0


**Висновок:** у typed стільки ж рядків, скільки в raw, нічого не відкинулось.

## Завдання 2. CTE-chain pipeline

Pipeline зберігаю в змінну `features_sql`, бо він потрібен ще для MV і EXPLAIN. Так весь SQL написаний один раз, і MV точно містить той самий pipeline.

In [10]:
features_sql = '''
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id
'''

Результат, топ-20 по monetary:

In [11]:
run(features_sql + '''
ORDER BY monetary DESC NULLS LAST, c.customer_id
LIMIT 20
''')

,customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0000,telefonia_fixa,13664.08
1,ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0000,telefonia_fixa,7274.88
2,c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0000,utilidades_domesticas,6929.31
3,f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0000,pcs,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0000,artes,6726.66
5,05455dfa7cd02f13d132aa7a6a9729c6,MG,328,1,6081.54,2017-11-24,2017-10-01,6081.54,0.0000,agro_industria_e_comercio,6081.54
6,df55c14d1476a9a3467f131269c2477f,RJ,565,1,4950.34,2017-04-01,2017-04-01,4950.34,0.0000,eletroportateis,4950.34
7,e0a2412720e9ea4f26c1ac985f6a7358,GO,98,1,4809.44,2018-07-12,2018-07-01,4809.44,1.0000,cool_stuff,4809.44
8,24bbf5fd2f2e1b359ee7de94defc4a15,SP,548,1,4764.34,2017-04-18,2017-04-01,4764.34,0.0000,eletroportateis,4764.34
9,3d979689f636322c62418b6346b1c6d2,PB,118,1,4681.78,2018-06-22,2018-04-01,4681.78,0.0000,pcs,4681.78


Перевірка окремих stages. Скільки рядків на кожному етапі і чи немає дублів customer_id після JOIN-ів:

In [12]:
stage_checks = features_sql.split('SELECT\n    c.customer_id,')[0] + '''
SELECT 'stage1_orders_total' AS stage, COUNT(*) AS n_rows, COUNT(DISTINCT customer_id) AS n_customers FROM stage1_orders_total
UNION ALL SELECT 'stage2_rfm', COUNT(*), COUNT(DISTINCT customer_id) FROM stage2_rfm
UNION ALL SELECT 'stage3_cohort', COUNT(*), COUNT(DISTINCT customer_id) FROM stage3_cohort
UNION ALL SELECT 'stage4_rolling_aov', COUNT(*), COUNT(DISTINCT customer_id) FROM stage4_rolling_aov
UNION ALL SELECT 'stage5_cancel_rate', COUNT(*), COUNT(DISTINCT customer_id) FROM stage5_cancel_rate
UNION ALL SELECT 'stage6_category_revenue', COUNT(*), COUNT(DISTINCT customer_id) FROM stage6_category_revenue
UNION ALL SELECT 'stage7_preferred_category', COUNT(*), COUNT(DISTINCT customer_id) FROM stage7_preferred_category
'''
run(stage_checks)

,stage,n_rows,n_customers
0,stage1_orders_total,99441,99441
1,stage2_rfm,99441,99441
2,stage3_cohort,99441,99441
3,stage4_rolling_aov,99441,99441
4,stage5_cancel_rate,99441,99441
5,stage6_category_revenue,99470,98666
6,stage7_preferred_category,98666,98666


Розподіл кількох ознак:

In [13]:
run('''
WITH f AS (
''' + features_sql + '''
)
SELECT
    COUNT(*) AS n_customers,
    MAX(frequency) AS max_frequency,
    ROUND(AVG(recency_days), 1) AS avg_recency,
    ROUND(AVG(cancellation_rate) * 100, 2) AS pct_canceled,
    COUNT(*) FILTER (WHERE preferred_category = '(none)') AS no_category,
    COUNT(*) FILTER (WHERE preferred_category = '(unknown)') AS unknown_category,
    COUNT(*) FILTER (WHERE avg_order_value_30d_recent <> monetary) AS aov_differs_from_monetary
FROM f
''')

,n_customers,max_frequency,avg_recency,pct_canceled,no_category,unknown_category,aov_differs_from_monetary
0,99441,1,291.3,0.63,775,1423,0


In [14]:
run('''
WITH f AS (
''' + features_sql + '''
)
SELECT cohort_quarter, COUNT(*) AS n_customers
FROM f
GROUP BY cohort_quarter
ORDER BY cohort_quarter
''')

,cohort_quarter,n_customers
0,2016-07-01,4
1,2016-10-01,325
2,2017-01-01,5262
3,2017-04-01,9349
4,2017-07-01,12642
5,2017-10-01,17848
6,2018-01-01,21208
7,2018-04-01,19979
8,2018-07-01,12820
9,2018-10-01,4


**Висновок:**
- Кожен stage рахує одну річ, і його можна перевірити окремо. Stages 1-5 мають по одному рядку на customer_id (99 441), тому фінальні LEFT JOIN нічого не дублюють. У stage6 99 470 рядків, бо у частини клієнтів кілька категорій, а stage7 знову зводить до одного рядка на клієнта.
- В Olist customer_id новий на кожне замовлення, тому frequency завжди 1, rolling AOV 30d дорівнює monetary, а cancellation_rate буває тільки 0 або 1 (0.63% скасованих). Pipeline працює правильно, але для реальної моделі ці ознаки краще рахувати по customer_unique_id. Тут залишаю customer_id, бо так вимагає завдання (UNIQUE INDEX на customer_id).
- 775 клієнтів мають preferred_category = `(none)`, бо їхні замовлення без позицій. Ще 1 423 мають `(unknown)`, це товари без категорії.
- Recency рахується від останньої дати в даних + 1 день, тому результат не змінюється з часом. Середня recency близько 291 дня.
- Найбільші когорти припадають на 2017-Q4 - 2018-Q2. 2016-Q3 (4) і 2018-Q4 (4) майже порожні, це краї датасету.

## Завдання 3. Recursive CTE для referral chain

Синтетична таблиця рефералів: перші 10 клієнтів по customer_id утворюють ланцюжок.

In [15]:
run("""
DROP TABLE IF EXISTS hw5_referrals CASCADE;

CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_at          TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    PRIMARY KEY (referred_customer_id),
    CHECK (referrer_customer_id <> referred_customer_id)
);

WITH ordered_customers AS (
    SELECT
        customer_id,
        ROW_NUMBER() OVER (ORDER BY customer_id) AS rn
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 10
)
INSERT INTO hw5_referrals (referrer_customer_id, referred_customer_id)
SELECT
    prev.customer_id AS referrer_customer_id,
    nxt.customer_id AS referred_customer_id
FROM ordered_customers AS prev
JOIN ordered_customers AS nxt
    ON nxt.rn = prev.rn + 1
ON CONFLICT (referred_customer_id) DO NOTHING;
""")

ok, rows affected: 9


In [16]:
run("""
SELECT referrer_customer_id, referred_customer_id
FROM hw5_referrals
ORDER BY referrer_customer_id, referred_customer_id;
""")

,referrer_customer_id,referred_customer_id
0,00012a2ce6f8dcda20d059ce98491703,000161a058600d5901f007fab4c27140
1,000161a058600d5901f007fab4c27140,0001fd6190edaaf884bcaf3d49edf079
2,0001fd6190edaaf884bcaf3d49edf079,0002414f95344307404f0ace7a26f1d5
3,0002414f95344307404f0ace7a26f1d5,000379cdec625522490c315e70c7a9fb
4,000379cdec625522490c315e70c7a9fb,0004164d20a9e969af783496f3408652
5,0004164d20a9e969af783496f3408652,000419c5494106c306a97b5635748086
6,000419c5494106c306a97b5635748086,00046a560d407e99b969756e0b10f282
7,00046a560d407e99b969756e0b10f282,00050bf6e01e69d5c0fd612f1bcfb69c
8,00050bf6e01e69d5c0fd612f1bcfb69c,000598caf2ef4117407665ac33275130


Recursive CTE: anchor через NOT EXISTS, захист від циклів через `path_ids` і `depth < 50`.

In [17]:
referral_sql = """
WITH RECURSIVE referral_depth AS (
    -- ANCHOR: root customers, які не були referred іншими customers
    SELECT
        c.customer_id,
        1 AS depth,
        ARRAY[c.customer_id]::TEXT[] AS path_ids,
        c.customer_id::TEXT AS path_text
    FROM olist_customers AS c
    WHERE NOT EXISTS (
        SELECT 1
        FROM hw5_referrals AS r
        WHERE r.referred_customer_id = c.customer_id
    )

    UNION ALL

    -- RECURSIVE: перейти від referrer до його referred customers
    SELECT
        r.referred_customer_id AS customer_id,
        rd.depth + 1 AS depth,
        rd.path_ids || r.referred_customer_id AS path_ids,
        rd.path_text || ' -> ' || r.referred_customer_id AS path_text
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd
        ON rd.customer_id = r.referrer_customer_id
    WHERE NOT r.referred_customer_id = ANY(rd.path_ids)
      AND rd.depth < 50
)
"""

run(referral_sql + """
SELECT
    depth,
    COUNT(*) AS n_customers,
    MAX(path_text) AS sample_path
FROM referral_depth
GROUP BY depth
ORDER BY depth;
""")

,depth,n_customers,sample_path
0,1,99432,ffffe8b65bbe3087b653a978c870db99
1,2,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
2,3,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
3,4,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
4,5,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
5,6,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
6,7,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
7,8,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
8,9,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
9,10,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...


Повний шлях для найглибшого клієнта (ID скорочую до 6 символів, щоб було видно весь ланцюжок):

In [18]:
run(referral_sql + """
SELECT
    depth,
    array_length(path_ids, 1) AS path_len,
    (SELECT STRING_AGG(LEFT(x, 6), ' -> ') FROM UNNEST(path_ids) AS x) AS short_path
FROM referral_depth
ORDER BY depth DESC
LIMIT 1;
""")

,depth,path_len,short_path
0,10,10,00012a -> 000161 -> 0001fd -> 000241 -> 000379...


Перевірка на циклі A -> B -> A (дані тут же в VALUES, таблицю не чіпаю). Старт з A:

In [19]:
run("""
WITH RECURSIVE edges(referrer, referred) AS (
    VALUES ('A', 'B'), ('B', 'A'), ('B', 'C')
),
walk AS (
    SELECT 'A'::TEXT AS node, 1 AS depth, ARRAY['A']::TEXT[] AS path_ids

    UNION ALL

    SELECT e.referred, w.depth + 1, w.path_ids || e.referred
    FROM edges AS e
    JOIN walk AS w
        ON w.node = e.referrer
    WHERE NOT e.referred = ANY(w.path_ids)
      AND w.depth < 50
)
SELECT node, depth, array_to_string(path_ids, ' -> ') AS path
FROM walk
ORDER BY depth, node;
""")

,node,depth,path
0,A,1,A
1,B,2,A -> B
2,C,3,A -> B -> C


**Висновок:** у ланцюжку 10 клієнтів, тому глибина від 1 до 10. На depth 1 лежать 99 432 кореневих клієнти: це всі, кого ніхто не запросив (99 441 - 9). На кожній глибині 2-10 по одному клієнту, і повний шлях має 10 вузлів.

Anchor зроблений через NOT EXISTS, а не NOT IN: якщо в підзапиті з'явиться NULL, NOT IN поверне порожній результат.

У тесті з циклом A -> B -> A запит зупинився: з B не пішов назад в A, бо A вже є в `path_ids`, і повернув тільки A, B, C. Без цієї перевірки рекурсія ходила б по колу, і тоді зупинив би тільки `depth < 50`.

## Завдання 4. Materialized View

MV будую з тієї самої змінної `features_sql`, тобто на повному pipeline.

In [20]:
run('DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE')
run('CREATE MATERIALIZED VIEW mv_olist_customer_features AS ' + features_sql)
run('''
CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id
    ON mv_olist_customer_features (customer_id)
''')

ok, rows affected: -1


ok, rows affected: 99441
ok, rows affected: -1


In [21]:
run("""
SELECT COUNT(*) AS n_rows
FROM mv_olist_customer_features;
""")

,n_rows
0,99441


In [22]:
run("""
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST, customer_id
LIMIT 5;
""")

,customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0000,telefonia_fixa,13664.08
1,ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0000,telefonia_fixa,7274.88
2,c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0000,utilidades_domesticas,6929.31
3,f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0000,pcs,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0000,artes,6726.66


Concurrent refresh:

In [23]:
run('REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features')

ok, rows affected: -1


Перевірка, що після refresh дані ті самі, що дає pipeline напряму:

In [24]:
run('''
WITH direct AS (
''' + features_sql + '''
)
SELECT
    (SELECT COUNT(*) FROM mv_olist_customer_features) AS mv_rows,
    (SELECT COUNT(*) FROM direct) AS pipeline_rows,
    (SELECT COUNT(*) FROM (
        SELECT * FROM direct
        EXCEPT
        SELECT * FROM mv_olist_customer_features
    ) AS diff) AS different_rows
''')

,mv_rows,pipeline_rows,different_rows
0,99441,99441,0


**Висновок:** MV має 99 441 рядок, по одному на кожного клієнта, і ті самі колонки, що й pipeline. `REFRESH ... CONCURRENTLY` пройшов без помилок, бо є UNIQUE INDEX на customer_id. Без нього PostgreSQL не може порівняти старі й нові рядки. Перевірка через EXCEPT показала 0 різних рядків між MV і прямим запуском pipeline.

## Завдання 5. EXPLAIN ANALYZE

### 5.1. Повний pipeline (всі stages, топ-10 по monetary)

In [25]:
explain(features_sql + '''
ORDER BY monetary DESC NULLS LAST
LIMIT 10
''')

Limit  (cost=97944.25..97944.28 rows=10 width=158) (actual time=2122.006..2125.921 rows=10 loops=1)
  Buffers: shared hit=12448, temp read=10036 written=10188
  CTE stage1_orders_total
    ->  Finalize GroupAggregate  (cost=16725.66..27526.12 rows=88592 width=124) (actual time=127.314..288.629 rows=99441 loops=1)
          Group Key: o_1.order_id
          Buffers: shared hit=4158, temp read=1546 written=1549
          ->  Gather Merge  (cost=16725.66..25488.91 rows=64913 width=136) (actual time=127.303..213.206 rows=99627 loops=1)
                Workers Planned: 1
                Workers Launched: 1
                Buffers: shared hit=4158, temp read=1546 written=1549
                ->  Partial GroupAggregate  (cost=15725.65..17186.19 rows=64913 width=136) (actual time=122.213..181.669 rows=49814 loops=2)
                      Group Key: o_1.order_id
                      Buffers: shared hit=4158, temp read=1546 written=1549
                      ->  Sort  (cost=15725.65..15887.93 r

### 5.2. Pipeline з шаблону (тільки до stage2_rfm)

In [26]:
explain(features_sql.split('stage3_cohort AS')[0].rstrip().rstrip(',') + '''
SELECT *
FROM stage2_rfm
ORDER BY monetary DESC NULLS LAST
LIMIT 10
''')

Limit  (cost=33552.08..33552.10 rows=10 width=64) (actual time=562.591..564.818 rows=10 loops=1)
  Buffers: shared hit=5966, temp read=3455 written=4193
  ->  Sort  (cost=33552.08..33552.58 rows=200 width=64) (actual time=562.589..564.814 rows=10 loops=1)
        Sort Key: stage2_rfm.monetary DESC NULLS LAST
        Sort Method: top-N heapsort  Memory: 27kB
        Buffers: shared hit=5966, temp read=3455 written=4193
        ->  Subquery Scan on stage2_rfm  (cost=33542.25..33547.75 rows=200 width=64) (actual time=405.736..540.887 rows=99441 loops=1)
              Buffers: shared hit=5966, temp read=3455 written=4193
              ->  HashAggregate  (cost=33542.25..33545.75 rows=200 width=68) (actual time=405.734..532.267 rows=99441 loops=1)
                    Group Key: o.customer_id, (((max(olist_orders.order_purchase_timestamp))::date + 1))
                    Batches: 21  Memory Usage: 8265kB  Disk Usage: 7400kB
                    Buffers: shared hit=5966, temp read=3455 written=

### 5.3. Lookup з MV по customer_id

In [27]:
explain('''
WITH target_customer AS (
    SELECT customer_id
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 1
)
SELECT mv.*
FROM mv_olist_customer_features AS mv
JOIN target_customer AS t
    ON t.customer_id = mv.customer_id
''')

Nested Loop  (cost=0.83..8.98 rows=1 width=158) (actual time=0.232..0.234 rows=1 loops=1)
  Buffers: shared hit=5 read=3
  ->  Limit  (cost=0.42..0.55 rows=1 width=32) (actual time=0.055..0.056 rows=1 loops=1)
        Buffers: shared hit=4
        ->  Index Only Scan using olist_customers_pkey on olist_customers  (cost=0.42..10897.92 rows=84100 width=32) (actual time=0.054..0.055 rows=1 loops=1)
              Heap Fetches: 1
              Buffers: shared hit=4
  ->  Index Scan using idx_mv_olist_customer_features_customer_id on mv_olist_customer_features mv  (cost=0.42..8.44 rows=1 width=158) (actual time=0.171..0.171 rows=1 loops=1)
        Index Cond: (customer_id = olist_customers.customer_id)
        Buffers: shared hit=1 read=3
Planning Time: 0.137 ms
Execution Time: 0.256 ms


### 5.4. Top-N з MV з індексом на monetary

Індекс роблю `DESC NULLS LAST`, як у запиті. Індекс з шаблону `(monetary DESC)` має порядок NULLS FIRST, і з `ORDER BY monetary DESC NULLS LAST` planner його не використовує (я перевіряв, був Seq Scan).

In [28]:
run('''
CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary
    ON mv_olist_customer_features (monetary DESC NULLS LAST)
''')

explain('''
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST
LIMIT 10
''')

ok, rows affected: -1
Limit  (cost=0.42..1.32 rows=10 width=158) (actual time=0.030..0.043 rows=10 loops=1)
  Buffers: shared hit=10 read=3
  ->  Index Scan using idx_mv_olist_customer_features_monetary on mv_olist_customer_features  (cost=0.42..8996.03 rows=99441 width=158) (actual time=0.029..0.041 rows=10 loops=1)
        Buffers: shared hit=10 read=3
Planning:
  Buffers: shared hit=15 read=1
Planning Time: 0.208 ms
Execution Time: 0.057 ms


### 5.5. Яка ознака найдорожча

Окремо міряю час кожної ознаки (той самий CTE, але SELECT тільки з потрібного stage). Кожен запит запускаю 3 рази і беру мінімум, бо час трохи скаче.

In [29]:
cte_part = features_sql.split('SELECT\n    c.customer_id,')[0]

feature_queries = {
    'base: stage1_orders_total': 'SELECT COUNT(*) FROM stage1_orders_total',
    'rfm + cohort (stage 1-3)': 'SELECT COUNT(*) FROM stage3_cohort',
    'rolling aov 30d (stage 4)': 'SELECT COUNT(*) FROM stage4_rolling_aov WHERE rn_recent = 1',
    'cancellation_rate (stage 5)': 'SELECT COUNT(*) FROM stage5_cancel_rate',
    'preferred_category (stage 6-7)': 'SELECT COUNT(*) FROM stage7_preferred_category',
}

times = []
for name, q in feature_queries.items():
    best = None
    for i in range(3):
        plan = run('EXPLAIN (ANALYZE, BUFFERS) ' + cte_part + q)
        lines = list(plan.iloc[:, 0])
        exec_line = [l for l in lines if l.startswith('Execution Time')][0]
        ms = float(exec_line.split()[2])
        if best is None or ms < best:
            best = ms
    times.append({'feature': name, 'execution_ms': best})

pd.DataFrame(times)

,feature,execution_ms
0,base: stage1_orders_total,369.135
1,rfm + cohort (stage 1-3),485.567
2,rolling aov 30d (stage 4),513.701
3,cancellation_rate (stage 5),512.665
4,preferred_category (stage 6-7),419.103


**Висновок:**
- Повний pipeline виконується близько 2 секунд. У плані видно Sort з `external merge` на диск, кілька WindowAgg і HashAggregate з `Batches: 21` та Disk Usage, тобто дані не влазять у work_mem (4 MB за замовчуванням) і пишуться на диск.
- Pipeline тільки до RFM приблизно в 4 рази швидший за повний.
- Lookup з MV по customer_id займає долі мілісекунди: це Index Scan по UNIQUE INDEX, який читає кілька сторінок.
- Top-N з MV з індексом `monetary DESC NULLS LAST` теж іде через Index Scan і читає тільки 10 рядків. Тобто MV швидша за pipeline в тисячі разів.
- По окремих ознаках: stage1 (сума замовлень) займає ~0.35 с і потрібен майже всім. RFM, rolling AOV і cancellation_rate додають до нього лише ~0.1-0.15 с. preferred_category не використовує stage1, але сама займає ~0.4 с, бо робить окремий join order_items + orders + products, Sort на диск, GroupAggregate і WindowAgg. Тому найдорожча ознака саме вона.

## Reflection

За EXPLAIN ANALYZE найдорожча ознака preferred_category. Вона не використовує stage1, а заново джойнить order_items, orders і products, потім робить Sort з external merge на диск, GroupAggregate і WindowAgg для ROW_NUMBER. Сама вона займає близько 0.4 с, тоді як RFM, rolling AOV і cancellation_rate додають до спільного stage1 лише ~0.1-0.15 с. Rolling AOV теж помітний, бо window function вимагає сортування по customer_id і часу, і цей Sort теж іде на диск. Ще в плані є HashAggregate з Batches: 21 і Disk Usage, тобто для прискорення можна збільшити work_mem.

Кешувати через MV варто: повний pipeline виконується ~2 с, а lookup з MV через Index Scan по UNIQUE INDEX займає долі мілісекунди. Нові замовлення в Olist не з'являються кожну хвилину, а модель зазвичай перенавчають раз на день, тому я б обрав щоденний REFRESH CONCURRENTLY вночі. Concurrently не блокує читання, тому модель може брати ознаки навіть під час оновлення.

Звичайний VIEW кращий, коли потрібні завжди свіжі дані і запит легкий, наприклад статус останнього замовлення клієнта для служби підтримки. Також VIEW має сенс, якщо даних мало і перерахунок дешевий.

Recursive CTE зупиняється, коли recursive part перестає повертати нові рядки. При циклі A -> B -> A без захисту рекурсія йшла б нескінченно. Тому є дві перевірки: `NOT next_id = ANY(path_ids)` не дає повернутись у вже відвіданий вузол, а `depth < 50` обмежує глибину. Я перевірив це на прикладі A -> B -> A, і запит зупинився на трьох вузлах. У PostgreSQL 14+ можна також використати CYCLE clause.

Для цього pipeline найважливіший trade-off між recompute-cost і freshness. Latency з MV і так дуже мала, а storage-cost невеликий (близько 100 тис. рядків). Головне питання в тому, як часто платити ~2 с за перерахунок, щоб ознаки не застаріли.